# Loan Approval Prediction

## Objective
Predict whether a loan should be approved using Decision Trees & Ensemble methods (Random Forest, XGBoost, LightGBM, CatBoost).

## Dataset
The dataset contains information about loan applicants, including gender, marital status, education, income, loan amount, credit history, etc.

## Steps
1. Data Loading and Exploration
2. Data Preprocessing (Handling Missing Values, Encoding)
3. Model Training (Decision Tree, Random Forest, XGBoost, LightGBM, CatBoost)
4. Model Evaluation
5. Feature Importance Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

# Set plot style
sns.set(style="whitegrid")

## 1. Data Loading and Exploration

In [ ]:
# Load dataset
df = pd.read_csv('dataset/load_predication_dataset.csv')

# Display first few rows
df.head()

In [ ]:
# Check dataset info
df.info()

In [ ]:
# Check for missing values
df.isnull().sum()

## 2. Data Preprocessing

In [ ]:
# Handling Missing Values

# For categorical variables, impute with mode
categorical_cols = ['Gender', 'Married', 'Dependents', 'Self_Employed', 'Credit_History', 'Loan_Amount_Term']
for col in categorical_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

# For numerical variables, impute with median
numerical_cols = ['LoanAmount']
for col in numerical_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Verify no missing values
df.isnull().sum().sum()

In [ ]:
# Exploratory Data Analysis (Viz target variable)
plt.figure(figsize=(6, 4))
sns.countplot(x='Loan_Status', data=df)
plt.title('Loan Status Distribution')
plt.show()

In [ ]:
# Convert 'Dependents' to numeric (replacing '3+' with 3)
df['Dependents'] = df['Dependents'].replace('3+', 3).astype(int)

# Encoding Categorical Variables
# Drop Loan_ID as it is not needed
df = df.drop(columns=['Loan_ID'])

# Label Encoding for categorical features
le = LabelEncoder()

cat_cols = ['Gender', 'Married', 'Education', 'Self_Employed', 'Property_Area', 'Loan_Status']
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

# Display processed dataframe info
df.info()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Split Data into X and y
X = df.drop(columns=['Loan_Status'])
y = df['Loan_Status']

# Split into Train and Test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

## 3. Model Training and Evaluation

In [ ]:
# Initialize models
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    "LightGBM": lgb.LGBMClassifier(random_state=42),
    "CatBoost": cb.CatBoostClassifier(verbose=0, random_state=42)
}

results = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f"{name} Accuracy: {acc:.4f}")
    print("-" * 30)

## 4. Model Comparison

In [ ]:
# Visualize Model Comparison
plt.figure(figsize=(10, 5))
sns.barplot(x=list(results.keys()), y=list(results.values()), palette='viridis')
plt.ylim(0.5, 1.0)
plt.ylabel('Accuracy')
plt.title('Model Accuracy Comparison')
plt.show()

## 5. Feature Importance (Random Forest)

In [ ]:
rf_model = models["Random Forest"]
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]
features = X.columns

plt.figure(figsize=(10, 6))
plt.title("Feature Importance (Random Forest)")
plt.bar(range(X.shape[1]), importances[indices], align="center")
plt.xticks(range(X.shape[1]), [features[i] for i in indices], rotation=45)
plt.tight_layout()
plt.show()

## 6. Confusion Matrix (Best Model)
Visualizing the confusion matrix for the Random Forest model (often a strong performer).

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

best_model_name = max(results, key=results.get)
print(f"Best Model: {best_model_name}")
best_model = models[best_model_name]

y_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No', 'Yes'])
disp.plot(cmap='Blues')
plt.title(f'Confusion Matrix - {best_model_name}')
plt.show()